In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/10/25 19:14:16] INFO     Found credentials from IAM Role:                                   ]8;id=632671;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=508002;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'gen-xiii-parser'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# preprocessing.py
COPY preprocessing.py ${LAMBDA_TASK_ROOT}

# cls_parser.pkl
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# counters
COPY functions_counters.py ${LAMBDA_TASK_ROOT}

# counters pricing
COPY functions_counters_pricing.py ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Load files from Flask app

In [4]:
list_str_filenames = [
    'api.py',
    'cls_parser.pkl',
    'preprocessing.py',
    'requirements.txt',
    'functions_counters.py',
    'functions_counters_pricing.py',
]
for str_filename in list_str_filenames:
    str_source = f'../07_flask_app/app/{str_filename}'
    str_destination = f'./{str_filename}'
    shutil.copyfile(str_source, str_destination)

### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pickle
import pandas as pd 
import json
pd.options.mode.chained_assignment = None # suppress warning

# lambda handler
def lambda_handler(event, context):
    # import parser
    print('Loading parser...')
    print('')
    cls_parser = pickle.load(open('cls_parser.pkl', 'rb'))
    # get payload
    print('Getting request...')
    print('')
    try:
        dict_json_request = event['request']
    except KeyError:
        dict_json_request = event
    str_json_request = json.dumps(dict_json_request)
    
    # parse payload
    print('Parsing payload...')
    cls_parser.get_data(str_request=str_json_request)
    cls_parser.engineer_pmt_hx()
    cls_parser.preprocessing()
    cls_parser.get_predictions()
    #cls_parser.interpolate()
    cls_parser.adverse_action()
    #cls_parser.counter_offers()
    cls_parser.generate_response()
    # extract output
    print('Extracting output...')
    print('')
    dict_response = cls_parser.dict_response
    print(dict_response)
    # return output_final
    return dict_response

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=gen-xiii-parser

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 764B done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.8
#2 DONE 0.2s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/10] FROM public.ecr.aws/lambda/python:3.8@sha256:19c611b6736ef5d1a1403a51b5140446c78158fb832771cac7be12ed513c361b
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.36MB done
#5 DONE 0.0s

#6 [ 2/10] RUN pip install --upgrade pip
#6 CACHED

#7 [ 3/10] COPY requirements.txt  .
#7 CACHED

#8 [ 4/10] RUN  pip3 install -r requirements.txt --target "/var/task"
#8 CACHED

#9 [ 5/10] COPY api.py /var/task
#9 DONE 0.0s

#10 [ 6/10] COPY preprocessing.py /var/task
#10 DONE 0.0s

#11 [ 7/10] COPY cls_parser.pkl /var/task
#11 DONE 0.0s

#12 [ 8/10] COPY functions_counters.py /var/task
#12 DONE 0.0s

#13 [ 9/10] COPY functions_counters_pricing.py /var/ta

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen-xiii-parser' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xiii-parser]
7fbcdce8d59e: Preparing
ca1874cccb6b: Preparing
7bf79dff6d7d: Preparing
6394ecf1156c: Preparing
2f9ee041bb5d: Preparing
6198f083c68d: Preparing
b58132fa9ca7: Preparing
b78f81525468: Preparing
e31fc74adc4e: Preparing
6198f083c68d: Waiting
69063223dcc9: Preparing
b58132fa9ca7: Waiting
3e0f7053d2d2: Preparing
b78f81525468: Waiting
e31fc74adc4e: Waiting
e1b8ef616f15: Preparing
884ba2d905e7: Preparing
7f2a4f045bc4: Preparing
3e0f7053d2d2: Waiting
e1b8ef616f15: Waiting
69063223dcc9: Waiting
814345d22610: Preparing
7f2a4f045bc4: Waiting
814345d22610: Waiting
7fbcdce8d59e: Pushed
ca1874cccb6b: Pushed
6394ecf1156c: Pushed
b58132fa9ca7: Layer already exists
b78f81525468: Layer already exists
e31fc74adc4e: Layer already exists
69063223dcc9: Layer already exists
3e0f7053d2d2: Layer already exists
2f9ee041bb5d: Pushed
e1b8ef616f15: Layer already exists
884ba2d905e7: Layer already exists
7f2a4f045bc4: Layer 

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

[03/10/25 19:14:21] INFO     Found credentials from IAM Role:                                   ]8;id=312317;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=224144;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 10 Mar 2025 19:14:22 GMT',
                                      'x-amzn-requestid': '57117dcb-a53a-4832-8683-236085404dd8'},
                      'HTTPStatusCode': 204,
                      'RequestId': '57117dcb-a53a-4832-8683-236085404dd8',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=180, # 3 minutes
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '513e934a0f60e87645fa0ea85b1d400be9bb728ec8192fb1ae0e08ae9d310bb8',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:gen-xiii-parser',
 'FunctionName': 'gen-xiii-parser',
 'LastModified': '2025-03-10T19:14:22.354+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/gen-xiii-parser'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1180',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 10 Mar 2025 19:14:23 GMT',
                                      'x-amzn-requestid': '7061cf70-962d-489a-afaf-2b60d8e4ea74'},
                      'HTTPStatusCode': 201,
                      'RequestId': '7061cf70-962d-489a-afaf-2b60d8e4ea74'

### Clean-up

In [11]:
list_str_filenames = [
    'requirements.txt',
    'preprocessing.py',
    'cls_parser.pkl',
    'Dockerfile',
    'api.py',
    'lambda_function.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
]
for str_file in list_str_filenames:
    try:
        os.remove(str_file)
    except:
        pass